# Gaz plivajućeg bloka — predvidi, izračunaj, provjeri

**Poglavlje U06: uzgon, plivanje i početni stabilitet**

Ravnotežni gaz pravokutnog bloka dobit ćemo i zatvorenom formulom i
numeričkim traženjem nule bilance sila. Zatim procjenjujemo osjetljivost na
masu, gustoću i dimenzije. Ovaj pokus ne ocjenjuje stabilitet ni sigurnost.


## 1. Predvidi

1. Hoće li isti blok imati veći gaz u slatkoj ili morskoj vodi?
2. Ako masa poraste 10 %, za koliko približno raste gaz?
3. Što se događa kada potrebni gaz dosegne visinu bloka $H$?

Prije računa nacrtaj težinu i silu uzgona te napiši predznake.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.dpi": 110, "font.size": 10})
G = 9.81

def gaz_analiticki(m, rho, L, B):
    return m/(rho*L*B)

def rezidual_sile(d, m, rho, L, B):
    return rho*G*L*B*d - m*G  # uzgon minus težina

m, rho, L, B, H = 1050.0, 1025.0, 2.0, 1.0, 0.80
d_ref = gaz_analiticki(m, rho, L, B)
print(f"Analitički gaz = {d_ref:.4f} m; nadvođe = {H-d_ref:.4f} m")


## 2. Izračunaj — gaz kao nula bilance sila

Bisekcija ne treba poznavati zatvorenu formulu. Ona traži $d$ za koji je
$F_B(d)-W=0$. Isti obrazac kasnije se koristi za složene oblike, tabličnu
hidrostatiku i nelinearne ravnoteže.


In [ ]:
def gaz_bisekcijom(m, rho, L, B, H, tol=1e-10, max_iter=100):
    a, b = 0.0, H
    if rezidual_sile(b, m, rho, L, B) < 0:
        raise ValueError("Tijelo s ovim podacima ne može plivati bez potapanja.")
    povijest = []
    for _ in range(max_iter):
        c = 0.5*(a+b)
        if rezidual_sile(c, m, rho, L, B) >= 0:
            b = c
        else:
            a = c
        povijest.append(0.5*(b-a))
        if 0.5*(b-a) < tol:
            break
    return 0.5*(a+b), np.asarray(povijest)

d_num, granica = gaz_bisekcijom(m, rho, L, B, H)
print(f"Numerički gaz = {d_num:.10f} m nakon {len(granica)} iteracija")

mase = np.linspace(500.0, 1600.0, 160)
fig, ax = plt.subplots(figsize=(7.0, 3.9))
for rho_f in (998.0, 1025.0):
    ax.plot(mase, gaz_analiticki(mase, rho_f, L, B),
            label=fr"$\rho_f={rho_f:.0f}$ kg/m³")
ax.axhline(H, color="#c62828", ls="--", label="visina bloka H")
ax.set(xlabel="masa (kg)", ylabel="gaz (m)", title="Parametarska osjetljivost gaza")
ax.grid(ls=":", alpha=0.6)
ax.legend()
plt.show()


## 3. Provjeri — ravnoteža i propagacija nesigurnosti

Za $d=m/(\rho LB)$ deterministička linearizacija kombinira nesigurnosti
mase, gustoće, duljine i širine. Rezultat je nesigurnost modeliranog gaza,
a ne sigurnosna margina plovila.


In [ ]:
u_m, u_rho, u_L, u_B = 3.0, 1.0, 0.005, 0.003
u_rel = np.sqrt((u_m/m)**2 + (u_rho/rho)**2 + (u_L/L)**2 + (u_B/B)**2)
u_d = d_ref*u_rel
debalans = rezidual_sile(d_num, m, rho, L, B)
print(f"d = {d_ref:.4f} ± {u_d:.4f} m")
print(f"debalans sile pri numeričkom gazu = {debalans:.3e} N")

assert abs(d_num-d_ref) < granica[-1] + 1e-12
assert abs(debalans) < 2e-6
m_potpuno = rho*L*B*H
assert np.isclose(gaz_analiticki(m_potpuno, rho, L, B), H, rtol=1e-12)
assert gaz_analiticki(m, 1025.0, L, B) < gaz_analiticki(m, 998.0, L, B)
print("PASS: bisekcija, bilanca uzgona, potpuno uranjanje i učinak gustoće.")


## Granica modela

Račun pretpostavlja mirnu vodu, jednoliku gustoću i pravokutni presjek bez
promjene oblika. Uvjet $d<H$ govori samo da postoji vertikalna ravnoteža s
nadvođem; ne dokazuje početni ni oštećeni stabilitet, preživljavanje ili
usklađenost s pomorskim pravilima.
